In [19]:
import json
from pathlib import Path

from filer_backend.filing.suggester import suggest_folders
from filer_backend.indexing.extract import extract_text

In [5]:
path_str = '/Users/ericmelz/Data/code/filer/apps/backend/evals/EXP03.json'
path = Path(path_str)
path

PosixPath('/Users/ericmelz/Data/code/filer/apps/backend/evals/EXP03.json')

In [6]:
with open(path, 'r') as f:
    data = json.load(f)

In [13]:
imperfect_pdfs = []
for s in data['samples']:
    if s['kind'] == 'pdf' and s['hit@1'] < 1.0:
        imperfect_pdfs.append(s)

In [14]:
len(imperfect_pdfs)

13

In [15]:
len(data['samples'])

50

In [16]:
total_pdfs = len([s for s in data['samples'] if s['kind']  == 'pdf'])
total_pdfs

43

In [17]:
pct_bad = len(imperfect_pdfs) / total_pdfs
pct_bad

0.3023255813953488

In [18]:
i0 = imperfect_pdfs[0]
i0

{'file_id': 286,
 'filename': '2025_08_31.pdf',
 'truth': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements',
 'predicted': ['/Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking',
  '/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements',
  '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements'],
 'kind': 'pdf',
 'density': 'existing',
 'hit@1': 0.0,
 'hit@3': 1.0,
 'rr': 0.3333333333333333,
 'prefix': 0.5833333333333334}

In [20]:
i0['truth']

'/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements'

In [24]:
text, parser = extract_text(Path(i0['truth']) / Path(i0['filename']))
print(text)

Initiate Business Checking SM 
August 31, 2025 Page 1 of 4 
Questions? 
Available by phone Mon-Sat 7:00am-11:00pm Eastern 
Time, Sun 9:00am-10:00pm Eastern Time: 
We accept all relay calls, including 711 
1-800-CALL-WELLS (1-800-225-5935) 
En español: 1-877-337-7454 
Online: wellsfargo.com/biz 
Write: Wells Fargo Bank, N.A. (114) 
P.O. Box 6995 
Portland, OR 97228-6995 
ERIC R MELZ 
RANDI M CURTIS 
DBA 15151 ENCANTO DRIVE 
300 S REEVES DR 
BEVERLY HILLS CA 90212-4513 
Your Business and Wells Fargo 
Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, 
infographics, and other resources on the topics of money movement, account 
management and monitoring, security and fraud prevention, and more.
Other Wells Fargo Benefits
You control your information - Be awarewhat you share
It could be something as innocent as your email address or where you bank or live. Be careful what you share and who you share
it with.
Fraudsters can use your personal information to steal your id

In [41]:
from filer_backend.eval.core import _to_inbox
from filer_backend.storage.db import get_session
from filer_backend.storage.models import File, Folder, InboxFile
from filer_backend.filing.suggester import suggest_folders
from sqlalchemy import select

In [27]:
s = get_session()

In [29]:
q = select(File)

In [30]:
q = q.where(File.id == i0['file_id'])

In [31]:
files = list(s.execute(q).scalars())

In [32]:
files

In [33]:
files[0]

In [35]:
files[0].id

286

In [36]:
f= files[0]

In [37]:
f.absolute_path

'/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements/2025_08_31.pdf'

In [38]:
inbf = _to_inbox(f)

In [39]:
inbf.id

'286'

In [43]:
# Jupyter already runs an asyncio event loop, so blocking calls like
# agent.run_sync() (used by suggest_folders) would otherwise raise
# "This event loop is already running". nest_asyncio makes the loop
# re-entrant so those sync helpers work inside the notebook.
import nest_asyncio

nest_asyncio.apply()

In [44]:
suggestions = suggest_folders(inbf, exclude_file_ids={f.id})

In [45]:
len(suggestions)

3

In [46]:
suggestions[0]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking', confidence=0.9, rationale='This folder contains similar files and is specifically for Wells Fargo Checking.', is_new=False)

In [47]:
suggestions[1]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements', confidence=0.8, rationale='This folder is relevant for Wells Fargo Checking statements and contains similar files.', is_new=False)

In [48]:
suggestions[2]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements', confidence=0.7, rationale='This folder is related to Wells Fargo statements and has similar files.', is_new=False)

Dive into suggestions inspired by suggester.py code

In [49]:
inbox_file = inbf

In [50]:
path = Path(inbox_file.absolute_path)

In [51]:
text, _ = extract_text(path)

In [53]:
from filer_backend.embedding import get_embedder

In [54]:
emb = get_embedder()

In [55]:
query = text[:4000]

In [56]:
obj = emb.embed([query])

In [58]:
len(obj)

1

In [59]:
qvec = obj[0]

In [60]:
exclude_file_ids = {inbox_file.id}
exclude_file_ids

{'286'}

In [61]:
from filer_backend.filing.retrieval import hybrid_search

In [62]:
hits = hybrid_search(query, qvec, k=20, exclude_file_ids=exclude_file_ids)

In [63]:
len(hits)

20

In [64]:
h0 = hits[0]

## Notice h0 - it's an Encanto statement, not a Whiffletree statement!